In [27]:
api_key = 'WxRSAcpuv90pVAdgPMmmQAAely5rpsJ6'

In [28]:
pip install arch


In [53]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from arch import arch_model

# Black-Scholes model
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

# GARCH volatility estimation
def estimate_garch_volatility(returns, lookback=126):
    scaled_returns = returns * 100
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="t")
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=30)
    daily_vol = np.sqrt(forecast.variance.iloc[-1].mean()) / 100
    annual_vol = daily_vol * np.sqrt(252)
    return annual_vol

# FMP API setup
API_KEY = api_key  # Replace with your FMP API key
BASE_URL = "https://financialmodelingprep.com/api/v3"

# Fetch AAPL historical data
def get_stock_data(symbol, start_date, end_date):
    url = f"{BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df[["close"]].rename(columns={"close": "Close"})
    return df.sort_index()

# Fetch data
stock_data = get_stock_data("AAPL", "2019-01-01", "2023-12-31")

# Calculate log returns
stock_data["Log Return"] = np.log(stock_data["Close"] / stock_data["Close"].shift(1))

# Initialize
capital = 10000
trades = []
commissions = 0.50
slippage = 0.0005
T = 30 / 365
dividend_yield = 0.009
rates = {2020: 0.005, 2021: 0.02, 2022: 0.04, 2023: 0.04}  # Annual risk-free rates
earnings_months = [2, 5, 8, 11]

# Resample to monthly
monthly_data = stock_data.resample("M").last()

for date, row in monthly_data[monthly_data.index >= "2020-01-31"].iterrows():
    if date.month in earnings_months:
        continue

    stock_price = row["Close"]
    year = date.year
    r = rates.get(year, 0.04)

    # Estimate GARCH volatility
    lookback_data = stock_data[stock_data.index < date].tail(126)["Log Return"].dropna()
    if len(lookback_data) < 126:
        continue
    sigma = estimate_garch_volatility(lookback_data)

    if sigma < 0.20:
        continue

    sigma_atm = sigma * 1.02
    S = stock_price * np.exp(-dividend_yield * T)

    # Short Straddle
    strike = round(stock_price)
    call_price = black_scholes(S, strike, T, r, sigma_atm, "call")
    put_price = black_scholes(S, strike, T, r, sigma_atm, "put")
    premium = (call_price + put_price) * 100 * (1 - slippage)
    trans_costs = 2 * commissions

    # Next month's stock price
    next_idx = monthly_data.index[monthly_data.index > date]
    if not next_idx.empty:
        next_month = monthly_data.loc[next_idx[0], "Close"]
    else:
        continue

    # Straddle Outcome
    straddle_value = max(strike - next_month, 0) + max(next_month - strike, 0)
    if straddle_value <= premium * 0.25 / 100:
        straddle_pnl = (premium - straddle_value * 100) - trans_costs
    elif straddle_value >= premium * 1.5 / 100:
        straddle_pnl = (premium - straddle_value * 100) - trans_costs
    else:
        straddle_pnl = (premium - straddle_value * 100) - trans_costs
    capital += straddle_pnl

    # Store risk-free rate for Sharpe
    monthly_rf = r / 12  # Monthly risk-free rate

    trades.append({
        "Date": date,
        "Stock Price": stock_price,
        "Volatility": sigma,
        "Strike": strike,
        "Premium": premium,
        "P/L": straddle_pnl,
        "Capital": capital,
        "Next Price": next_month,
        "Monthly RF": monthly_rf
    })

# Results
trades_df = pd.DataFrame(trades)
trades_df["Return %"] = (trades_df["P/L"] / trades_df["Capital"].shift(1)) * 100

# Calculate Sharpe Ratio
trades_df["Excess Return"] = trades_df["Return %"] / 100 - trades_df["Monthly RF"]
mean_excess_return = trades_df["Excess Return"].mean()
std_returns = trades_df["Return %"].std() / 100
sharpe_ratio = (mean_excess_return * 12) / (std_returns * np.sqrt(12)) if std_returns > 0 else 0

print(trades_df[["Date", "Stock Price", "Volatility", "Strike", "Premium", "P/L", "Next Price", "Capital", "Return %"]])
total_return = (trades_df["Capital"].iloc[-1] - 10000) / 10000 * 100
annualized_return = ((trades_df["Capital"].iloc[-1] / 10000) ** (1 / 4) - 1) * 100
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Max Drawdown: {((trades_df['Capital'].cummax() - trades_df['Capital']) / trades_df['Capital'].cummax() * 100).max():.2f}%")
print(f"Win Rate: {(trades_df['P/L'] > 0).mean() * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

<ipython-input-53-881d44511f35>:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = stock_data.resample("M").last()


         Date  Stock Price  Volatility  Strike      Premium          P/L  \
0  2020-03-31        63.57    0.812197      64  1205.145240   259.145240   
1  2020-04-30        73.45    0.444960      73   759.615849   109.615849   
2  2020-06-30        91.20    0.502012      91  1065.037666  -461.962334   
3  2020-07-31       106.26    0.432372     106  1068.943729 -1236.056271   
4  2020-09-30       115.81    0.405109     116  1093.874426   378.874426   
5  2020-10-31       108.86    0.544604     109  1381.271166   375.271166   
6  2020-12-31       132.69    0.416165     133  1288.054895  1183.054895   
7  2021-01-31       131.96    0.394260     132  1211.368752   136.368752   
8  2021-03-31       122.15    0.335299     122   953.250527     6.250527   
9  2021-04-30       131.46    0.289298     131   885.233087   245.233087   
10 2021-07-31       145.86    0.236683     146   804.349521   220.349521   
11 2021-12-31       177.57    0.225922     178   935.639401   612.639401   
12 2022-01-3

In [54]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from arch import arch_model

# Black-Scholes model
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

# GARCH volatility estimation
def estimate_garch_volatility(returns, lookback=126):
    scaled_returns = returns * 100
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="t")
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=30)
    daily_vol = np.sqrt(forecast.variance.iloc[-1].mean()) / 100
    annual_vol = daily_vol * np.sqrt(252)
    return annual_vol

# FMP API setup
API_KEY = api_key  # Replace with your FMP API key
BASE_URL = "https://financialmodelingprep.com/api/v3"

# Fetch AAPL historical data
def get_stock_data(symbol, start_date, end_date):
    url = f"{BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df[["close"]].rename(columns={"close": "Close"})
    return df.sort_index()

# Fetch data
stock_data = get_stock_data("AAPL", "2019-01-01", "2023-12-31")

# Calculate log returns
stock_data["Log Return"] = np.log(stock_data["Close"] / stock_data["Close"].shift(1))

# Initialize
capital = 10000
shares = 100
trades = []
commissions = 0.50  # $0.50/contract
slippage = 0.0005  # 0.05% slippage
stock_comm = 0.01  # $0.01/share
T = 30 / 365
dividend_yield = 0.009
rates = {2020: 0.005, 2021: 0.02, 2022: 0.04, 2023: 0.04}
earnings_months = [2, 5, 8, 11]

# Resample to monthly
monthly_data = stock_data.resample("M").last()

for date, row in monthly_data[monthly_data.index >= "2020-01-31"].iterrows():
    if date.month in earnings_months:
        continue

    stock_price = row["Close"]
    year = date.year
    r = rates.get(year, 0.04)

    # Estimate GARCH volatility
    lookback_data = stock_data[stock_data.index < date].tail(126)["Log Return"].dropna()
    if len(lookback_data) < 126:
        continue
    sigma = estimate_garch_volatility(lookback_data)

    if sigma < 0.20:
        continue

    sigma_atm = sigma * 1.02  # ATM skew
    sigma_otm = sigma * 1.05  # OTM put skew
    S = stock_price * np.exp(-dividend_yield * T)

    # Buy 100 shares if not held
    if not trades or trades[-1]["Shares"] == 0:
        cost = stock_price * shares + shares * stock_comm
        capital -= cost
    else:
        shares = trades[-1]["Shares"]
        cost = 0

    # Short Straddle
    strike = round(stock_price)
    call_price = black_scholes(S, strike, T, r, sigma_atm, "call")
    put_price = black_scholes(S, strike, T, r, sigma_atm, "put")
    straddle_premium = (call_price + put_price) * 100 * (1 - slippage)

    # Protective Put
    net_premium = straddle_premium
    breakeven = stock_price - net_premium / 100
    put_strike = round(breakeven * 0.90 / 5) * 5
    put_price = black_scholes(S, put_strike, T, r, sigma_otm, "put")
    put_premium = put_price * 100 * (1 + slippage)  # Buy at ask
    net_premium = straddle_premium - put_premium

    # Transaction costs
    trans_costs = 3 * commissions  # 2 straddle contracts + 1 put

    # Next month's stock price
    next_idx = monthly_data.index[monthly_data.index > date]
    if not next_idx.empty:
        next_month = monthly_data.loc[next_idx[0], "Close"]
    else:
        continue

    # Straddle Outcome
    straddle_value = max(strike - next_month, 0) + max(next_month - strike, 0)
    if straddle_value <= straddle_premium * 0.25 / 100:
        straddle_pnl = (straddle_premium - straddle_value * 100) - trans_costs
    elif straddle_value >= straddle_premium * 1.5 / 100:
        straddle_pnl = (straddle_premium - straddle_value * 100) - trans_costs
    else:
        straddle_pnl = (straddle_premium - straddle_value * 100) - trans_costs

    # Stock P/L
    if next_month > strike:
        stock_pnl = (strike - stock_price) * shares - shares * stock_comm
        shares = 0
    elif next_month < put_strike:
        stock_pnl = (put_strike - stock_price) * shares - shares * stock_comm
        shares = 0
    else:
        stock_pnl = (next_month - stock_price) * shares

    # Protective Put P/L
    put_pnl = max(put_strike - next_month, 0) * 100 - put_premium

    # Total P/L
    net_pnl = stock_pnl + straddle_pnl + put_pnl
    capital += net_pnl

    # Monthly risk-free rate for Sharpe
    monthly_rf = r / 12

    trades.append({
        "Date": date,
        "Stock Price": stock_price,
        "Volatility": sigma,
        "Strike": strike,
        "Put Strike": put_strike,
        "Straddle Premium": straddle_premium,
        "Put Premium": put_premium,
        "Next Price": next_month,
        "Breakeven": breakeven,
        "Shares": shares,
        "Stock P/L": stock_pnl,
        "Straddle P/L": straddle_pnl,
        "Put P/L": put_pnl,
        "Net P/L": net_pnl,
        "Capital": capital,
        "Monthly RF": monthly_rf
    })

# Results
trades_df = pd.DataFrame(trades)
trades_df["Return %"] = (trades_df["Net P/L"] / trades_df["Capital"].shift(1)) * 100
trades_df["Excess Return"] = trades_df["Return %"] / 100 - trades_df["Monthly RF"]
mean_excess_return = trades_df["Excess Return"].mean()
std_returns = trades_df["Return %"].std() / 100
sharpe_ratio = (mean_excess_return * 12) / (std_returns * np.sqrt(12)) if std_returns > 0 else 0

print(trades_df[["Date", "Stock Price", "Volatility", "Strike", "Put Strike", "Breakeven",
                 "Next Price", "Stock P/L", "Straddle P/L", "Put P/L", "Net P/L", "Capital", "Return %"]])
total_return = (trades_df["Capital"].iloc[-1] - 10000) / 10000 * 100
annualized_return = ((trades_df["Capital"].iloc[-1] / 10000) ** (1 / 4) - 1) * 100
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Max Drawdown: {((trades_df['Capital'].cummax() - trades_df['Capital']) / trades_df['Capital'].cummax() * 100).max():.2f}%")
print(f"Win Rate: {(trades_df['Net P/L'] > 0).mean() * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

<ipython-input-54-5f5f7227e757>:61: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = stock_data.resample("M").last()


         Date  Stock Price  Volatility  Strike  Put Strike   Breakeven  \
0  2020-03-31        63.57    0.812197      64          45   51.518548   
1  2020-04-30        73.45    0.444960      73          60   65.853842   
2  2020-06-30        91.20    0.502012      91          70   80.549623   
3  2020-07-31       106.26    0.432372     106          85   95.570563   
4  2020-09-30       115.81    0.405109     116          95  104.871256   
5  2020-10-31       108.86    0.544604     109          85   95.047288   
6  2020-12-31       132.69    0.416165     133         110  119.809451   
7  2021-01-31       131.96    0.394260     132         110  119.846312   
8  2021-03-31       122.15    0.335299     122         100  112.617495   
9  2021-04-30       131.46    0.289298     131         110  122.607669   
10 2021-07-31       145.86    0.236683     146         125  137.816505   
11 2021-12-31       177.57    0.225922     178         150  168.213606   
12 2022-01-31       174.78    0.318516

In [55]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from arch import arch_model

# Black-Scholes model
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

# GARCH volatility estimation
def estimate_garch_volatility(returns, lookback=126):
    scaled_returns = returns * 100
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="t")
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=30)
    daily_vol = np.sqrt(forecast.variance.iloc[-1].mean()) / 100
    annual_vol = daily_vol * np.sqrt(252)
    return annual_vol

# FMP API setup
API_KEY = api_key  # Replace with your FMP API key
BASE_URL = "https://financialmodelingprep.com/api/v3"

# Fetch AAPL historical data
def get_stock_data(symbol, start_date, end_date):
    url = f"{BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df[["close"]].rename(columns={"close": "Close"})
    return df.sort_index()

# Fetch data
stock_data = get_stock_data("AAPL", "2019-01-01", "2023-12-31")

# Calculate log returns
stock_data["Log Return"] = np.log(stock_data["Close"] / stock_data["Close"].shift(1))

# Initialize
capital = 10000
trades = []
commissions = 0.50
slippage = 0.0005
T = 30 / 365
dividend_yield = 0.009
rates = {2020: 0.005, 2021: 0.02, 2022: 0.04, 2023: 0.04}
earnings_months = [2, 5, 8, 11]

# Resample to monthly
monthly_data = stock_data.resample("M").last()

for date, row in monthly_data[monthly_data.index >= "2020-01-31"].iterrows():
    if date.month in earnings_months:
        continue

    stock_price = row["Close"]
    year = date.year
    r = rates.get(year, 0.04)

    # Estimate GARCH volatility
    lookback_data = stock_data[stock_data.index < date].tail(126)["Log Return"].dropna()
    if len(lookback_data) < 126:
        continue
    sigma = estimate_garch_volatility(lookback_data)

    # Decide long or short straddle
    if sigma < 0.20:  # Long straddle
        trade_type = "long"
    elif sigma > 0.30:  # Short straddle
        trade_type = "short"
    else:
        continue

    sigma_atm = sigma * 1.02
    S = stock_price * np.exp(-dividend_yield * T)

    # Straddle
    strike = round(stock_price)
    call_price = black_scholes(S, strike, T, r, sigma_atm, "call")
    put_price = black_scholes(S, strike, T, r, sigma_atm, "put")
    if trade_type == "long":
        premium = (call_price + put_price) * 100 * (1 + slippage)  # Buy at ask
    else:
        premium = (call_price + put_price) * 100 * (1 - slippage)  # Sell at bid
    trans_costs = 2 * commissions

    # Next month's stock price
    next_idx = monthly_data.index[monthly_data.index > date]
    if not next_idx.empty:
        next_month = monthly_data.loc[next_idx[0], "Close"]
    else:
        continue

    # Straddle Outcome
    straddle_value = max(strike - next_month, 0) + max(next_month - strike, 0)
    if trade_type == "long":
        if straddle_value >= premium * 1.25 / 100:  # 25% profit
            straddle_pnl = (straddle_value * 100 - premium) - trans_costs
        elif straddle_value <= premium * 0.6667 / 100:  # 1.5x loss
            straddle_pnl = (straddle_value * 100 - premium) - trans_costs
        else:
            straddle_pnl = (straddle_value * 100 - premium) - trans_costs
    else:
        if straddle_value <= premium * 0.25 / 100:
            straddle_pnl = (premium - straddle_value * 100) - trans_costs
        elif straddle_value >= premium * 1.5 / 100:
            straddle_pnl = (premium - straddle_value * 100) - trans_costs
        else:
            straddle_pnl = (premium - straddle_value * 100) - trans_costs
    capital += straddle_pnl

    monthly_rf = r / 12
    trades.append({
        "Date": date,
        "Stock Price": stock_price,
        "Volatility": sigma,
        "Trade Type": trade_type,
        "Strike": strike,
        "Premium": premium,
        "P/L": straddle_pnl,
        "Capital": capital,
        "Next Price": next_month,
        "Monthly RF": monthly_rf
    })

# Results
trades_df = pd.DataFrame(trades)
trades_df["Return %"] = (trades_df["P/L"] / trades_df["Capital"].shift(1)) * 100
trades_df["Excess Return"] = trades_df["Return %"] / 100 - trades_df["Monthly RF"]
mean_excess_return = trades_df["Excess Return"].mean()
std_returns = trades_df["Return %"].std() / 100
sharpe_ratio = (mean_excess_return * 12) / (std_returns * np.sqrt(12)) if std_returns > 0 else 0

print(trades_df[["Date", "Stock Price", "Volatility", "Trade Type", "Strike", "Premium", "P/L", "Next Price", "Capital", "Return %"]])
total_return = (trades_df["Capital"].iloc[-1] - 10000) / 10000 * 100
annualized_return = ((trades_df["Capital"].iloc[-1] / 10000) ** (1 / 4) - 1) * 100
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Max Drawdown: {((trades_df['Capital'].cummax() - trades_df['Capital']) / trades_df['Capital'].cummax() * 100).max():.2f}%")
print(f"Win Rate: {(trades_df['P/L'] > 0).mean() * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

<ipython-input-55-527077c9118c>:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = stock_data.resample("M").last()


         Date  Stock Price  Volatility Trade Type  Strike      Premium  \
0  2020-01-31        77.38    0.182676       long      77   330.152045   
1  2020-03-31        63.57    0.812197      short      64  1205.145240   
2  2020-04-30        73.45    0.444960      short      73   759.615849   
3  2020-06-30        91.20    0.502012      short      91  1065.037666   
4  2020-07-31       106.26    0.432372      short     106  1068.943729   
5  2020-09-30       115.81    0.405109      short     116  1093.874426   
6  2020-10-31       108.86    0.544604      short     109  1381.271166   
7  2020-12-31       132.69    0.416165      short     133  1288.054895   
8  2021-01-31       131.96    0.394260      short     132  1211.368752   
9  2021-03-31       122.15    0.335299      short     122   953.250527   
10 2021-06-30       136.96    0.185555       long     137   592.593668   
11 2021-09-30       141.50    0.197825       long     142   654.399500   
12 2021-10-31       149.80    0.192615

In [46]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from arch import arch_model

# Black-Scholes model
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

# GARCH volatility estimation
def estimate_garch_volatility(returns, lookback=126):
    scaled_returns = returns * 100
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="t")
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=30)
    daily_vol = np.sqrt(forecast.variance.iloc[-1].mean()) / 100
    annual_vol = daily_vol * np.sqrt(252)
    return annual_vol

# FMP API setup
API_KEY = api_key  # Replace with your FMP API key
BASE_URL = "https://financialmodelingprep.com/api/v3"

# Fetch AAPL historical data
def get_stock_data(symbol, start_date, end_date):
    url = f"{BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df[["close"]].rename(columns={"close": "Close"})
    return df.sort_index()

# Fetch data
stock_data = get_stock_data("KO", "2019-01-01", "2023-12-31")

# Calculate log returns
stock_data["Log Return"] = np.log(stock_data["Close"] / stock_data["Close"].shift(1))

# Initialize
capital = 10000
trades = []
commissions = 0.50
slippage = 0.0005
T = 30 / 365
dividend_yield = 0.009
rates = {2020: 0.005, 2021: 0.02, 2022: 0.04, 2023: 0.04}
earnings_months = [2, 5, 8, 11]

# Resample to monthly
monthly_data = stock_data.resample("M").last()

for date, row in monthly_data[monthly_data.index >= "2020-01-31"].iterrows():
    if date.month in earnings_months:
        continue

    stock_price = row["Close"]
    year = date.year
    r = rates.get(year, 0.04)

    # Estimate GARCH volatility
    lookback_data = stock_data[stock_data.index < date].tail(126)["Log Return"].dropna()
    if len(lookback_data) < 126:
        continue
    sigma = estimate_garch_volatility(lookback_data)

    if sigma <= 0.20:
        continue

    sigma_sell = sigma * 1.02
    sigma_buy = sigma * 1.05
    S = stock_price * np.exp(-dividend_yield * T)

    # Iron Condor
    call_strike = round(stock_price * 1.05)
    call_strike_high = call_strike + 5
    put_strike = round(stock_price * 0.95)
    put_strike_low = put_strike - 5

    # Sell call, buy call
    call_sell = black_scholes(S, call_strike, T, r, sigma_sell, "call") * 100 * (1 - slippage)
    call_buy = black_scholes(S, call_strike_high, T, r, sigma_buy, "call") * 100 * (1 + slippage)
    call_spread_premium = call_sell - call_buy

    # Sell put, buy put
    put_sell = black_scholes(S, put_strike, T, r, sigma_sell, "put") * 100 * (1 - slippage)
    put_buy = black_scholes(S, put_strike_low, T, r, sigma_buy, "put") * 100 * (1 + slippage)
    put_spread_premium = put_sell - put_buy

    net_premium = call_spread_premium + put_spread_premium
    trans_costs = 4 * commissions

    # Next month's stock price
    next_idx = monthly_data.index[monthly_data.index > date]
    if not next_idx.empty:
        next_month = monthly_data.loc[next_idx[0], "Close"]
    else:
        continue

    # Iron Condor Outcome
    call_spread_value = max(next_month - call_strike, 0) - max(next_month - call_strike_high, 0)
    put_spread_value = max(put_strike - next_month, 0) - max(put_strike_low - next_month, 0)
    ic_value = (call_spread_value + put_spread_value)

    if ic_value <= net_premium * 0.25 / 100:
        ic_pnl = (net_premium - ic_value * 100) - trans_costs
    elif ic_value >= net_premium * 1.5 / 100:
        ic_pnl = (net_premium - ic_value * 100) - trans_costs
    else:
        ic_pnl = (net_premium - ic_value * 100) - trans_costs
    capital += ic_pnl

    monthly_rf = r / 12
    trades.append({
        "Date": date,
        "Stock Price": stock_price,
        "Volatility": sigma,
        "Call Strike": call_strike,
        "Call High Strike": call_strike_high,
        "Put Strike": put_strike,
        "Put Low Strike": put_strike_low,
        "Net Premium": net_premium,
        "P/L": ic_pnl,
        "Capital": capital,
        "Next Price": next_month,
        "Monthly RF": monthly_rf
    })

# Results
trades_df = pd.DataFrame(trades)
trades_df["Return %"] = (trades_df["P/L"] / trades_df["Capital"].shift(1)) * 100
trades_df["Excess Return"] = trades_df["Return %"] / 100 - trades_df["Monthly RF"]
mean_excess_return = trades_df["Excess Return"].mean()
std_returns = trades_df["Return %"].std() / 100
sharpe_ratio = (mean_excess_return * 12) / (std_returns * np.sqrt(12)) if std_returns > 0 else 0

print(trades_df[["Date", "Stock Price", "Volatility", "Call Strike", "Call High Strike",
                 "Put Strike", "Put Low Strike", "Net Premium", "P/L", "Next Price", "Capital", "Return %"]])
total_return = (trades_df["Capital"].iloc[-1] - 10000) / 10000 * 100
annualized_return = ((trades_df["Capital"].iloc[-1] / 10000) ** (1 / 4) - 1) * 100
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Max Drawdown: {((trades_df['Capital'].cummax() - trades_df['Capital']) / trades_df['Capital'].cummax() * 100).max():.2f}%")
print(f"Win Rate: {(trades_df['P/L'] > 0).mean() * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

<ipython-input-46-983ae6df2a6d>:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = stock_data.resample("M").last()


         Date  Stock Price  Volatility  Call Strike  Call High Strike  \
0  2020-03-31        44.25    0.829073           46                51   
1  2020-04-30        45.89    0.316307           48                53   
2  2020-06-30        44.68    0.397418           47                52   
3  2020-07-31        47.24    0.404776           50                55   
4  2020-09-30        49.37    0.211621           52                57   
5  2020-12-31        54.84    0.227969           58                63   
6  2021-01-31        48.15    0.233330           51                56   
7  2021-03-31        52.71    0.206204           55                60   
8  2022-04-30        64.61    0.236088           68                73   
9  2022-06-30        62.91    0.284481           66                71   
10 2022-07-31        64.17    0.221544           67                72   
11 2022-09-30        56.02    0.214047           59                64   
12 2022-10-31        59.85    0.209061           63

In [52]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import norm
from arch import arch_model

# Black-Scholes model
def black_scholes(S, K, T, r, sigma, option_type="put"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "put":
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price

# GARCH volatility estimation
def estimate_garch_volatility(returns, lookback=126):
    scaled_returns = returns * 100
    model = arch_model(scaled_returns, vol="Garch", p=1, q=1, dist="t")
    res = model.fit(disp="off")
    forecast = res.forecast(horizon=30)
    daily_vol = np.sqrt(forecast.variance.iloc[-1].mean()) / 100
    annual_vol = daily_vol * np.sqrt(252)
    return annual_vol

# FMP API setup
API_KEY = api_key  # Replace with your FMP API key
BASE_URL = "https://financialmodelingprep.com/api/v3"

# Fetch AAPL historical data
def get_stock_data(symbol, start_date, end_date):
    url = f"{BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={API_KEY}"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df[["close"]].rename(columns={"close": "Close"})
    return df.sort_index()

# Fetch data
stock_data = get_stock_data("KO", "2019-01-01", "2023-12-31")

# Calculate log returns
stock_data["Log Return"] = np.log(stock_data["Close"] / stock_data["Close"].shift(1))

# Initialize
capital = 10000
shares = 0
trades = []
commissions = 0.50
slippage = 0.0005
stock_comm = 0.01
T = 30 / 365
dividend_yield = 0.009
rates = {2020: 0.005, 2021: 0.02, 2022: 0.04, 2023: 0.04}
earnings_months = [2, 5, 8, 11]

# Resample to monthly
monthly_data = stock_data.resample("M").last()

for date, row in monthly_data[monthly_data.index >= "2020-01-31"].iterrows():
    stock_price = row["Close"]
    year = date.year
    r = rates.get(year, 0.04)

    # Buy 100 shares if none held
    if shares == 0:
        cost = stock_price * 100 + 100 * stock_comm
        if capital >= cost:
            capital -= cost
            shares = 100
        else:
            continue

    # Estimate GARCH volatility
    lookback_data = stock_data[stock_data.index < date].tail(126)["Log Return"].dropna()
    if len(lookback_data) < 126:
        continue
    sigma = estimate_garch_volatility(lookback_data)

    # Protective Put (skip during earnings months)
    put_premium = 0
    put_strike = 0
    put_pnl = 0
    trans_costs = 0
    if date.month not in earnings_months:
        sigma_otm = sigma * 1.05
        S = stock_price * np.exp(-dividend_yield * T)
        put_strike = round(stock_price * 0.90 / 5) * 5
        put_price = black_scholes(S, put_strike, T, r, sigma_otm, "put")
        put_premium = put_price * 100 * (1 + slippage)
        trans_costs = commissions
        capital -= put_premium + trans_costs

    # Next month's stock price
    next_idx = monthly_data.index[monthly_data.index > date]
    if not next_idx.empty:
        next_month = monthly_data.loc[next_idx[0], "Close"]
    else:
        continue

    # Stock P/L
    if put_strike > 0 and next_month < put_strike:
        stock_pnl = (put_strike - stock_price) * shares - shares * stock_comm
        capital += put_strike * shares  # Cash from put exercise
        shares = 0
    else:
        stock_pnl = (next_month - stock_price) * shares

    # Put P/L
    if put_strike > 0:
        put_pnl = max(put_strike - next_month, 0) * 100 - put_premium

    # Total P/L
    net_pnl = stock_pnl + put_pnl
    capital += net_pnl

    monthly_rf = r / 12
    trades.append({
        "Date": date,
        "Stock Price": stock_price,
        "Volatility": sigma,
        "Put Strike": put_strike,
        "Put Premium": put_premium,
        "Next Price": next_month,
        "Shares": shares,
        "Stock P/L": stock_pnl,
        "Put P/L": put_pnl,
        "Net P/L": net_pnl,
        "Capital": capital,
        "Monthly RF": monthly_rf
    })

# Results
trades_df = pd.DataFrame(trades)
trades_df["Return %"] = (trades_df["Net P/L"] / trades_df["Capital"].shift(1)) * 100
trades_df["Excess Return"] = trades_df["Return %"] / 100 - trades_df["Monthly RF"]
mean_excess_return = trades_df["Excess Return"].mean()
std_returns = trades_df["Return %"].std() / 100
sharpe_ratio = (mean_excess_return * 12) / (std_returns * np.sqrt(12)) if std_returns > 0 else 0

print(trades_df[["Date", "Stock Price", "Volatility", "Put Strike", "Put Premium",
                 "Next Price", "Stock P/L", "Put P/L", "Net P/L", "Capital", "Return %"]])
total_return = (trades_df["Capital"].iloc[-1] - 10000) / 10000 * 100
annualized_return = ((trades_df["Capital"].iloc[-1] / 10000) ** (1 / 4) - 1) * 100
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Max Drawdown: {((trades_df['Capital'].cummax() - trades_df['Capital']) / trades_df['Capital'].cummax() * 100).max():.2f}%")
print(f"Win Rate: {(trades_df['Net P/L'] > 0).mean() * 100:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")

<ipython-input-52-2c952f3d60f7>:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_data = stock_data.resample("M").last()


         Date  Stock Price  Volatility  Put Strike  Put Premium  Next Price  \
0  2020-01-31        58.40    0.113006          55     3.089787       53.49   
1  2020-02-29        53.49    0.203381           0     0.000000       44.25   
2  2020-03-31        44.25    0.829073          40   239.856728       45.89   
3  2020-04-30        45.89    0.316307          40    13.700087       46.68   
4  2020-05-31        46.68    0.300011           0     0.000000       44.68   
5  2020-06-30        44.68    0.397418          40    48.707987       47.24   
6  2020-07-31        47.24    0.404776          45   130.113676       49.53   
7  2020-08-31        49.53    0.148694           0     0.000000       49.37   
8  2020-09-30        49.37    0.211621          45     9.854578       48.06   
9  2020-10-31        48.06    0.180384          45    14.003460       51.60   
10 2020-11-30        51.60    0.225387           0     0.000000       54.84   
11 2020-12-31        54.84    0.227969          50  